# RDD Sensitivity: Bandwidth and Polynomial Order

**Module 05 | Estimated time: 15 minutes**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Implement a full bandwidth sensitivity analysis for RDD
2. Compare estimates across polynomial orders (1, 2, 3)
3. Implement and interpret the donut RDD robustness test
4. Run placebo cutoff tests
5. Produce a comprehensive sensitivity results table

## Why Sensitivity Matters

An RDD result should not depend critically on arbitrary choices like bandwidth size. A credible study shows that estimates are stable across a reasonable range of bandwidths, that placebo cutoffs give null results, and that the donut RDD (excluding the innermost observations) still recovers the effect.

In [ ]:
learning_objectives(["Implement a full bandwidth sensitivity analysis for RDD", "Compare estimates across polynomial orders (1, 2, 3)", "Implement and interpret the donut RDD robustness test", "Run placebo cutoff tests", "Produce a comprehensive sensitivity results table"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import statsmodels.formula.api as smf
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(1234)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# Recreate scholarship RDD dataset from notebook 01
N = 2000
CUTOFF = 0  # working with centred x throughout
TRUE_EFFECT = 0.25

x = np.random.normal(0, 60, N)
x = np.clip(x, -250, 250)
treated = (x >= 0).astype(int)
gpa = (2.8 + 0.008*x - 0.000015*x**2
       + TRUE_EFFECT * treated
       + np.random.normal(0, 0.3, N))
gpa = np.clip(gpa, 0, 4.0)

df = pd.DataFrame({'x': x, 'treated': treated, 'gpa': gpa})
print(f"Dataset: N={N}, true effect = {TRUE_EFFECT}")

## 1. Local Linear Regression Helper

We use a flexible estimation function that supports bandwidth, polynomial order, and optional donut.

In [ ]:
def rdd_estimate(df, outcome, x_col, bandwidth, poly_order=1, donut=0.0):
    """
    Estimate sharp RDD.
    
    Parameters
    ----------
    bandwidth : float — half-width of estimation window
    poly_order : int — polynomial order for running variable (1 or 2)
    donut : float — exclude observations within this distance of cutoff
    """
    # Select observations within bandwidth
    mask = np.abs(df[x_col]) <= bandwidth
    if donut > 0:
        mask = mask & (np.abs(df[x_col]) > donut)
    local_df = df[mask].copy()
    
    if local_df.shape[0] < 10:
        return None
    
    local_df['treated'] = (local_df[x_col] >= 0).astype(int)
    
    # Build formula
    if poly_order == 1:
        formula = f'{outcome} ~ treated + {x_col} + treated:{x_col}'
    elif poly_order == 2:
        local_df['x2'] = local_df[x_col] ** 2
        formula = f'{outcome} ~ treated + {x_col} + x2 + treated:{x_col} + treated:x2'
    else:
        local_df['x2'] = local_df[x_col] ** 2
        local_df['x3'] = local_df[x_col] ** 3
        formula = f'{outcome} ~ treated + {x_col} + x2 + x3 + treated:{x_col} + treated:x2 + treated:x3'
    
    try:
        model = smf.ols(formula, data=local_df).fit(cov_type='HC1')
        tau = model.params['treated']
        se = model.bse['treated']
        ci = model.conf_int().loc['treated'].values
        return {'estimate': tau, 'se': se, 'ci_lo': ci[0], 'ci_hi': ci[1],
                'n': int(mask.sum())}
    except Exception:
        return None

# Test it
r = rdd_estimate(df, 'gpa', 'x', bandwidth=100)
print(f"Local linear estimate at bandwidth=100: τ = {r['estimate']:.3f} (SE={r['se']:.3f})")

## 2. Bandwidth Sensitivity Analysis

In [ ]:
bandwidths = np.arange(20, 220, 10)
sens_results = []

for h in bandwidths:
    r = rdd_estimate(df, 'gpa', 'x', bandwidth=h, poly_order=1)
    if r:
        sens_results.append({'bandwidth': h, **r})

sens_df = pd.DataFrame(sens_results)

# Plot bandwidth sensitivity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: estimates + CI across bandwidths
ax = axes[0]
ax.plot(sens_df['bandwidth'], sens_df['estimate'], 'o-',
        color='steelblue', linewidth=2, markersize=5, label='Estimate')
ax.fill_between(sens_df['bandwidth'],
                sens_df['ci_lo'],
                sens_df['ci_hi'],
                alpha=0.2, color='steelblue', label='95% CI')
ax.axhline(TRUE_EFFECT, color='green', linestyle='--', linewidth=2,
           label=f'True effect = {TRUE_EFFECT}')
ax.axhline(0, color='black', linestyle='-', alpha=0.3, linewidth=0.8)

ax.set_xlabel('Bandwidth h')
ax.set_ylabel('Treatment Effect Estimate')
ax.set_title('Bandwidth Sensitivity: Local Linear RDD')
ax.legend()
ax.grid(alpha=0.2)

# Right: effective sample size
ax2 = axes[1]
ax2.plot(sens_df['bandwidth'], sens_df['n'], 'o-',
         color='darkorange', linewidth=2, markersize=5)
ax2.set_xlabel('Bandwidth h')
ax2.set_ylabel('Effective Sample Size (within bandwidth)')
ax2.set_title('Sample Size vs Bandwidth')
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print("Bandwidth Sensitivity Summary:")
print(f"  Estimate range: [{sens_df['estimate'].min():.3f}, {sens_df['estimate'].max():.3f}]")
print(f"  All CIs exclude zero: {(sens_df['ci_lo'] > 0).all()}")
print(f"  Estimate variation: {sens_df['estimate'].std():.3f}")
print(f"\nConclusion: estimates are {'stable' if sens_df['estimate'].std() < 0.05 else 'sensitive'} across bandwidths")

## 3. Polynomial Order Sensitivity

In [ ]:
poly_results = []
h_fixed = 120  # fix bandwidth for polynomial comparison

for p in [1, 2, 3]:
    r = rdd_estimate(df, 'gpa', 'x', bandwidth=h_fixed, poly_order=p)
    if r:
        poly_results.append({'order': p, **r})

poly_df = pd.DataFrame(poly_results)

print(f"Polynomial Order Comparison (bandwidth = {h_fixed}):")
print("=" * 55)
print(f"{'Order':<8} {'Estimate':>10} {'SE':>8} {'95% CI':>20}")
print("-" * 55)
for _, row in poly_df.iterrows():
    ci_str = f"[{row['ci_lo']:+.3f}, {row['ci_hi']:+.3f}]"
    print(f"p={int(row['order']):<7} {row['estimate']:>+10.3f} {row['se']:>8.3f} {ci_str:>20}")
print("-" * 55)
print(f"{'True effect':<8} {TRUE_EFFECT:>+10.3f}")

# Visualise polynomial fit
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (_, row) in zip(axes, poly_df.iterrows()):
    p = int(row['order'])
    local_df = df[np.abs(df['x']) <= h_fixed].copy()
    local_df['treated'] = (local_df['x'] >= 0).astype(int)
    
    # Scatter
    left = local_df[local_df['x'] < 0]
    right = local_df[local_df['x'] >= 0]
    ax.scatter(left['x'], left['gpa'], alpha=0.2, s=8, color='steelblue')
    ax.scatter(right['x'], right['gpa'], alpha=0.2, s=8, color='darkorange')
    
    # Fitted polynomial
    for side_df, color in [(left, 'steelblue'), (right, 'darkorange')]:
        coeffs = np.polyfit(side_df['x'], side_df['gpa'], p)
        x_fit = np.linspace(side_df['x'].min(), side_df['x'].max(), 200)
        y_fit = np.polyval(coeffs, x_fit)
        ax.plot(x_fit, y_fit, '-', color=color, linewidth=2.5)
    
    ax.axvline(0, color='red', ls='--', linewidth=1.5)
    ax.set_title(f'Polynomial order p={p}\nτ = {row["estimate"]:+.3f}')
    ax.set_xlabel('x (centred)')
    ax.set_ylabel('GPA' if p == 1 else '')
    ax.set_ylim(2.0, 3.8)

plt.suptitle('Polynomial Order Comparison: Same Data, Different Fits', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Donut RDD

In [ ]:
donut_results = []
h_donut = 120

for donut_width in [0, 2, 5, 10, 15, 20]:
    r = rdd_estimate(df, 'gpa', 'x', bandwidth=h_donut, poly_order=1, donut=donut_width)
    if r:
        donut_results.append({'donut': donut_width, **r})

donut_df = pd.DataFrame(donut_results)

print(f"Donut RDD Results (bandwidth = {h_donut}):")
print("=" * 60)
print(f"{'Donut Width':>12} {'Estimate':>10} {'SE':>8} {'N obs':>7} {'95% CI':>20}")
print("-" * 60)
for _, row in donut_df.iterrows():
    ci_str = f"[{row['ci_lo']:+.3f}, {row['ci_hi']:+.3f}]"
    print(f"{row['donut']:>12.0f} {row['estimate']:>+10.3f} {row['se']:>8.3f} "
          f"{row['n']:>7} {ci_str:>20}")
print("-" * 60)

fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(donut_df['donut'], donut_df['estimate'],
            yerr=[donut_df['estimate'] - donut_df['ci_lo'],
                  donut_df['ci_hi'] - donut_df['estimate']],
            fmt='o-', capsize=5, color='steelblue', linewidth=2, markersize=7)
ax.axhline(TRUE_EFFECT, color='green', ls='--', linewidth=2, label=f'True effect = {TRUE_EFFECT}')
ax.axhline(0, color='black', ls='-', alpha=0.3)
ax.set_xlabel('Donut Width (excluded on each side of cutoff)')
ax.set_ylabel('Estimated Treatment Effect')
ax.set_title('Donut RDD: Stability Test for Local Manipulation')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Placebo Cutoff Test

In [ ]:
# Test RDD at false cutoffs (using only the control side — below the true cutoff)
control_df = df[df['x'] < 0].copy()
placebo_cutoffs = [-100, -80, -60, -40, -20, -10]

placebo_results = []
for c_placebo in placebo_cutoffs:
    # Re-centre around placebo cutoff
    control_df['x_placebo'] = control_df['x'] - c_placebo
    r = rdd_estimate(control_df, 'gpa', 'x_placebo', bandwidth=60, poly_order=1)
    if r:
        placebo_results.append({'cutoff': c_placebo, **r})

placebo_df = pd.DataFrame(placebo_results)

print("Placebo Cutoff Test (estimates should be near zero):")
print("=" * 55)
print(f"{'Placebo Cutoff':>15} {'Estimate':>10} {'p-value':>10}")
print("-" * 55)
for _, row in placebo_df.iterrows():
    t_stat = row['estimate'] / row['se']
    p_val = 2 * (1 - stats.norm.cdf(abs(t_stat)))
    flag = " ← significant" if p_val < 0.05 else ""
    print(f"{row['cutoff']:>15} {row['estimate']:>+10.3f} {p_val:>10.3f}{flag}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(placebo_df['cutoff'], placebo_df['estimate'],
            yerr=[placebo_df['estimate'] - placebo_df['ci_lo'],
                  placebo_df['ci_hi'] - placebo_df['estimate']],
            fmt='o', capsize=5, color='steelblue', linewidth=2, markersize=8,
            label='Placebo estimate')
ax.axhline(0, color='black', ls='-', linewidth=0.8)
ax.axhline(TRUE_EFFECT, color='green', ls='--', linewidth=2, alpha=0.5,
           label=f'Real effect = {TRUE_EFFECT}')
ax.set_xlabel('Placebo Cutoff (centred units)')
ax.set_ylabel('RDD Estimate at Placebo Cutoff')
ax.set_title('Placebo Test: No Effect at False Cutoffs')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Comprehensive Sensitivity Table

In [ ]:
print("=" * 65)
print("TABLE: RDD SENSITIVITY ANALYSIS — MERIT SCHOLARSHIP ON GPA")
print("=" * 65)
print(f"{'Specification':<35} {'Estimate':>9} {'SE':>7} {'95% CI':>18}")
print("-" * 65)

specs = [
    # Primary
    ("Primary: Local linear, h=100", 'gpa', 'x', 100, 1, 0),
    # Bandwidth variation
    ("Narrow: h=60", 'gpa', 'x', 60, 1, 0),
    ("Wide: h=150", 'gpa', 'x', 150, 1, 0),
    # Polynomial order
    ("Quadratic, h=100", 'gpa', 'x', 100, 2, 0),
    # Donut
    ("Donut h=100, d=10", 'gpa', 'x', 100, 1, 10),
    ("Donut h=100, d=20", 'gpa', 'x', 100, 1, 20),
]

for label, outcome, x_col, h, p, donut in specs:
    r = rdd_estimate(df, outcome, x_col, bandwidth=h, poly_order=p, donut=donut)
    if r:
        ci_str = f"[{r['ci_lo']:+.3f}, {r['ci_hi']:+.3f}]"
        print(f"{label:<35} {r['estimate']:>+9.3f} {r['se']:>7.3f} {ci_str:>18}")

print("-" * 65)
print(f"{'True effect':<35} {TRUE_EFFECT:>+9.3f}")
print("=" * 65)
print("\nAll specifications give estimates close to the true effect.")
print("All 95% CIs exclude zero. Result is robust.")

## Self-Check

1. Set `TRUE_EFFECT = 0.05` (a small effect). At what bandwidth does the 95% CI start to include zero? What does this tell you about minimum detectable effects in RDD?

2. Increase the noise to `sigma=0.8` and rerun the full sensitivity analysis. Do the estimates remain stable? Do the placebo tests still pass?

3. Create a scenario where the donut RDD gives a very different estimate from the main estimate. Under what data generating process would this happen?

4. Run placebo tests on the treated side (x > 0) rather than the control side. Do they behave similarly? When might they not?

---
**Next:** `03_causalpy_rdd.ipynb` — CausalPy's `RegressionDiscontinuity` class

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])